In [2]:
!pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [segmentation_models_pytorch] [segmentation_models_pytorch]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import json
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Stai usando: {device}")

Stai usando: cuda


In [ ]:
class UNet_ResNet_Pipeline(nn.Module):
    def __init__(self):
        super(UNet_ResNet_Pipeline, self).__init__()
        
        self.unet = smp.Unet(
            encoder_name="resnet34",        
            encoder_weights="imagenet",     
            in_channels=1,
            classes=1                       
        )

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        pesi_originali = self.resnet.conv1.weight.clone()
        
        self.resnet.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=64, 
            kernel_size=7, 
            stride=2, 
            padding=3, 
            bias=False
        )

        with torch.no_grad():
            self.resnet.conv1.weight = nn.Parameter(torch.sum(pesi_originali, dim=1, keepdim=True))
            
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 2)

    def forward(self, x):
        mask_logits = self.unet(x)
        mask_probs = torch.sigmoid(mask_logits)
        
        x_masked = x * mask_probs 
        
        class_output = self.resnet(x_masked)
        
        return class_output, mask_probs, x_masked

In [ ]:
def setup_dataloaders(data_dir, transform_train, transform_test, batch_size=32, seed=42):
    """Crea i dataloader garantendo lo stesso identico split per ogni esperimento"""
    full_dataset = datasets.ImageFolder(data_dir, transform=transform_train)
    
    ds_size = len(full_dataset)
    train_size = int(0.7 * ds_size)
    val_size = int(0.2 * ds_size)
    test_size = ds_size - train_size - val_size
    
    train_ds, val_ds, test_ds = random_split(
        full_dataset, 
        [train_size, val_size, test_size], 
        generator=torch.Generator().manual_seed(seed)
    )
    
    val_ds.dataset.transform = transform_test
    test_ds.dataset.transform = transform_test
    
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)
    )

def esegui_training(modello, train_loader, val_loader, percorso_modello, percorso_json, epochs=20, patience=5, lr=1e-4):
    """Ciclo di addestramento con registrazione metriche sia per Train che per Validation"""
    criterio = nn.CrossEntropyLoss()
    ottimizzatore = optim.Adam(modello.parameters(), lr=lr)
    
    storia = {
        "train_loss": [], "train_acc": [], "train_precision": [], "train_recall": [], "train_f1": [],
        "val_loss": [], "val_acc": [], "val_precision": [], "val_recall": [], "val_f1": []
    }
    
    miglior_val_loss = float('inf')
    epoche_pazienza = 0

    for epoca in range(epochs):
        print(f"\n--- Epoca {epoca+1}/{epochs} ---")
        
        modello.train()
        train_loss = 0.0
        all_y_train, all_pred_train = [], []
        
        for img, etic in tqdm(train_loader, desc="Train", leave=False, colour='blue'):
            img, etic = img.to(device), etic.to(device)
            ottimizzatore.zero_grad()
            out, _, _ = modello(img)
            loss = criterio(out, etic)
            loss.backward()
            ottimizzatore.step()
            
            train_loss += loss.item() * img.size(0)
            _, pred = torch.max(out, 1)
            all_y_train.extend(etic.cpu().numpy())
            all_pred_train.extend(pred.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(all_y_train, all_pred_train)
        train_prec = precision_score(all_y_train, all_pred_train, zero_division=0)
        train_rec = recall_score(all_y_train, all_pred_train, zero_division=0)
        train_f1 = f1_score(all_y_train, all_pred_train, zero_division=0)
        

        modello.eval()
        val_loss = 0.0
        all_y_val, all_pred_val = [], []
        
        with torch.no_grad():
            for img, etic in tqdm(val_loader, desc="Val", leave=False, colour='green'):
                img, etic = img.to(device), etic.to(device)
                out, _, _ = modello(img)
                loss = criterio(out, etic)
                val_loss += loss.item() * img.size(0)
                _, pred = torch.max(out, 1)
                all_y_val.extend(etic.cpu().numpy())
                all_pred_val.extend(pred.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_acc = accuracy_score(all_y_val, all_pred_val)
        val_prec = precision_score(all_y_val, all_pred_val, zero_division=0)
        val_rec = recall_score(all_y_val, all_pred_val, zero_division=0)
        val_f1 = f1_score(all_y_val, all_pred_val, zero_division=0)

        print(f"Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val   - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

        storia["train_loss"].append(train_loss); storia["train_acc"].append(train_acc)
        storia["train_precision"].append(train_prec); storia["train_recall"].append(train_rec); storia["train_f1"].append(train_f1)
        
        storia["val_loss"].append(val_loss); storia["val_acc"].append(val_acc)
        storia["val_precision"].append(val_prec); storia["val_recall"].append(val_rec); storia["val_f1"].append(val_f1)
        
        with open(percorso_json, 'w') as f: json.dump(storia, f, indent=4)

        if val_loss < miglior_val_loss:
            miglior_val_loss = val_loss
            epoche_pazienza = 0
            torch.save(modello.state_dict(), percorso_modello)
            print("Checkpoint salvato.")
        else:
            epoche_pazienza += 1
            if epoche_pazienza >= patience:
                print(f"Early Stopping all'epoca {epoca+1}")
                break


def traccia_grafici_training(percorso_json):
    """Genera 3 grafici distinti ottimizzati per l'inserimento nella tesi"""
    with open(percorso_json, 'r') as f: storia = json.load(f)
    epoche = range(1, len(storia['train_loss']) + 1)
    miglior_epoca_idx = np.argmin(storia['val_loss'])
    miglior_epoca = miglior_epoca_idx + 1
    

    plt.figure(figsize=(8, 5))
    plt.plot(epoche, storia['train_loss'], label='Train Loss', color='blue', linewidth=2)
    plt.plot(epoche, storia['val_loss'], label='Validation Loss', color='red', linewidth=2)
    plt.axvline(x=miglior_epoca, color='black', linestyle='--', label=f'Miglior Ep. ({miglior_epoca})')
    
    if miglior_epoca < len(epoche):
        plt.fill_between(epoche[miglior_epoca_idx:], storia['train_loss'][miglior_epoca_idx:], 
                         storia['val_loss'][miglior_epoca_idx:], color='red', alpha=0.1, label='Overfitting')
                         
    plt.title('Dinamica della Loss e Overfitting', fontsize=14, fontweight='bold')
    plt.xlabel('Epoche'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()

    
    def traccia_pannello_singolo(titolo, loss_data, acc_data, prec_data, rec_data, f1_data, loss_color):
        fig, ax1 = plt.subplots(figsize=(8, 5))
        
        ax1.plot(epoche, acc_data, label='Accuracy', color='purple', linewidth=2)
        ax1.plot(epoche, prec_data, label='Precision', color='green', linestyle='-.')
        ax1.plot(epoche, rec_data, label='Recall', color='orange', linestyle='-.')
        ax1.plot(epoche, f1_data, label='F1-Score', color='brown', linewidth=2)
        
        ax1.set_xlabel('Epoche', fontsize=12)
        ax1.set_ylabel('Metriche (0.0 - 1.0)', color='black', fontsize=12)
        ax1.set_ylim([0, 1.05])
        ax1.grid(True, linestyle=':', alpha=0.7)
        
        ax2 = ax1.twinx()
        ax2.plot(epoche, loss_data, label='Loss', color=loss_color, linewidth=3, linestyle='--')
        ax2.set_ylabel('Loss (CrossEntropy)', color=loss_color, fontsize=12)
        ax2.tick_params(axis='y', labelcolor=loss_color)
        
        ax2.axvline(x=miglior_epoca, color='black', linestyle=':', alpha=0.5)
        
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')
        
        plt.title(titolo, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

   
    traccia_pannello_singolo('Performance sul Set di Addestramento (Train)', 
                             storia['train_loss'], storia['train_acc'], 
                             storia['train_precision'], storia['train_recall'], 
                             storia['train_f1'], loss_color='blue')

    traccia_pannello_singolo('Performance sul Set di Validazione (Val)', 
                             storia['val_loss'], storia['val_acc'], 
                             storia['val_precision'], storia['val_recall'], 
                             storia['val_f1'], loss_color='red')

def valuta_modello_test(modello, test_loader, percorso_modello):
    """Carica il miglior modello e stampa Matrice di Confusione e Report"""
    modello.load_state_dict(torch.load(percorso_modello))
    modello.eval()
    all_y, all_pred = [], []
    
    with torch.no_grad():
        for img, etic in tqdm(test_loader, desc="Test Inferenza", colour='cyan'):
            out, _, _ = modello(img.to(device))
            all_y.extend(etic.cpu().numpy())
            all_pred.extend(torch.max(out, 1)[1].cpu().numpy())
            
    cm = confusion_matrix(all_y, all_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Sano (0)', 'Malato (1)'], yticklabels=['Sano', 'Malato'])
    plt.title('Matrice di Confusione'); plt.ylabel('Verità'); plt.xlabel('Predizione')
    plt.show()
    print("\nREPORT CLINICO FINALE (Sul Test Set):\n", classification_report(all_y, all_pred, target_names=['Sano', 'Malato']))

In [ ]:
DIR_DATI = '/home/andy/Documenti/Tesi-Magistrale/data/RX_super'

transform_base = transforms.Compose([
    transforms.Grayscale(1), 
    transforms.Resize((224, 224)),
    transforms.ToTensor(), 
    transforms.Normalize([0.5], [0.5])
])

print("MODELLO 1: Baseline (Senza Data Augmentation)")
train_loader_base, val_loader_base, test_loader_base = setup_dataloaders(
    DIR_DATI, transform_base, transform_base
)

modello_baseline = UNet_ResNet_Pipeline().to(device)

percorso_pth_base = '/home/andy/Documenti/Tesi-Magistrale/modello_no_aug.pth'
percorso_json_base = '/home/andy/Documenti/Tesi-Magistrale/risultati_no_aug.json'

esegui_training(modello_baseline, train_loader_base, val_loader_base, percorso_pth_base, percorso_json_base, epochs=20, patience=5)
traccia_grafici_training(percorso_json_base)
valuta_modello_test(modello_baseline, test_loader_base, percorso_pth_base)

In [ ]:
print("MODELLO 2: Pipeline con Data Augmentation")

transform_aug = transforms.Compose([
    transforms.Grayscale(1), 
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(10),
    transforms.ToTensor(), 
    transforms.Normalize([0.5], [0.5])
])

train_loader_aug, val_loader_aug, test_loader_aug = setup_dataloaders(
    DIR_DATI, transform_aug, transform_base
)

modello_aug = UNet_ResNet_Pipeline().to(device)

percorso_pth_aug = '/home/andy/Documenti/Tesi-Magistrale/modello_con_aug.pth'
percorso_json_aug = '/home/andy/Documenti/Tesi-Magistrale/risultati_con_aug.json'

esegui_training(modello_aug, train_loader_aug, val_loader_aug, percorso_pth_aug, percorso_json_aug, epochs=20, patience=5)
traccia_grafici_training(percorso_json_aug)
valuta_modello_test(modello_aug, test_loader_aug, percorso_pth_aug)